In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as dataset

import numpy as np
import matplotlib.pyplot as plt

In [2]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [3]:
class EducationalMoE(nn.Module):
    """
    Minimal Mixture of Experts for learning purposes.
    Shows the core concepts without complexity.
    """
    def __init__(self, d_model, num_experts=4, top_k=2):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k

        # Create simple experts (just linear layers for learning)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.ReLU(),
                nn.Linear(d_model, d_model)
            ) for _ in range(num_experts)
        ])

        # Router - decides which expert to use
        self.router = nn.Linear(d_model, num_experts)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, d_model)
        """
        # Step 1: Router computes score for each expert
        router_logits = self.router(x)  # router_logits is (B, T, num_experts)
        router_probs = F.softmax(router_logits, dim=-1) # router_probs is (B, T, num_experts)

        # Step 2: Select top-k experts for each token; top_k_probs, top_k_indices are (B,T,top_k)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)

        # Step 3: Normalize top-k probabilities (they sum to 1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        self.aux_loss = self.compute_load_balancing_loss(router_probs, top_k_indices)

        # Step 4: Initialize output
        output = torch.zeros_like(x)

        # Step 5: For each expert, apply to tokens that chose it
        for expert_idx, expert in enumerate(self.experts):
            # Find which tokens selected this expert; expert_mask is (B,T)
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)

            if expert_mask.any():
                # Get tokens that need this expert
                # expert_input is (N,D),
                # N — number of tokens that selected this expert (i.e., number of True values in expert_mask)
                expert_input = x[expert_mask]

                # Apply expert
                # expert_output is (N,D)
                expert_output = expert(expert_input)

                # Get corresponding weights
                # Find positions in top_k where this expert was chosen
                # Input: top_k_indices has shape (B,T,top_k)
                # Output: Boolean tensor weight_mask of shape (B,T,top_k)
                # True — the expert at this position in the top‑k list is expert_idx
                # False — otherwise
                weight_mask = (top_k_indices == expert_idx)

                # top_k_probs — tensor of shape (B,T,top_k) containing probabilities for the top‑k experts per token
                # weight_mask — boolean tensor of the same shape (B,T,top_k)
                # Operation: Boolean indexing. Selects all elements from top_k_probs where the corresponding element in weight_mask is True
                # Output of top_k_probs[weight_mask] (before .unsqueeze(-1)) is 1D tensor containing the probabilities of expert_idx for all tokens that selected it
                # .unsqueeze(-1):
                #     Input: 1D tensor of shape (N,) from the previous step
                #     Operation: .unsqueeze(-1) adds a new dimension at the end
                #     Output: Tensor weights of shape (N,1)
                weights = top_k_probs[weight_mask].unsqueeze(-1)

                # Weighted sum
                # output	(B,T,D) - Final output tensor: B sequences, each with T tokens, each token represented as a D-dimensional vector
                # expert_mask	(B,T) - Boolean mask: True at position (b,t) means token t in batch b was assigned to this expert
                # expert_output	(N,D) - Output from the current expert: N tokens (those selected by the mask) processed through the expert; each is a D-dimensional vector
                # weights	(N,1) - Routing weights (probabilities) for the N tokens assigned to this expert; each weight is a scalar
                # expert_output * weights	(N,D)	Weighted expert output: each of the N token vectors is scaled by its routing weight
                # output[expert_mask]	(N,D)	Slice of output corresponding to the N masked tokens; shape matches the weighted expert output
                output[expert_mask] += expert_output * weights

        return output

    def compute_load_balancing_loss(self, router_probs, top_k_indices):
        """Calculate auxiliary loss to encourage balanced expert usage"""
        # Get which experts were selected for each token
        # top_k_indices shape: (batch_size, seq_len, top_k)

        # Count how many times each expert was selected
        expert_mask = F.one_hot(top_k_indices, num_classes=self.num_experts)  # (B, T, top_k, num_experts)
        # which specific experts were chosen for each token into a single, aggregated count per token
        expert_mask = expert_mask.sum(dim=-2)  # Sum over top_k dimension -> (B, T, num_experts)
        expert_mask = expert_mask.float()

        # Fraction of tokens routed to each expert
        fraction_per_expert = expert_mask.mean(dim=(0, 1))  # Average over batch and sequence

        # Average routing probability for each expert
        # router_probs shape: (B, T, num_experts)
        avg_prob_per_expert = router_probs.mean(dim=(0, 1))

        # Load balancing loss (encourage fraction ≈ probability)
        # Lower is better (balanced when fraction_per_expert = avg_prob_per_expert)
        aux_loss = torch.sum(fraction_per_expert * avg_prob_per_expert) * self.num_experts

        return aux_loss

    def get_aux_loss(self):
        return self.aux_loss

    def get_expert_usage(self, x):
        """Debugging: See which experts are being used"""
        with torch.no_grad():
            router_logits = self.router(x)
            router_probs = F.softmax(router_logits, dim=-1)
            top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)[1]

            usage = {}
            for i in range(self.num_experts):
                usage[f'expert_{i}'] = (top_k_indices == i).sum().item()
            return usage

In [4]:
class DecoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.ln3 = nn.LayerNorm(d_model)
    self.mha1 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=True)
    self.mha2 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):
    # self-attention on decoder input
    x = self.ln1(dec_input + self.mha1(dec_input, dec_input, dec_input, dec_mask))

    # multi-head attention including incoder output
    x = self.ln2(x + self.mha2(x, enc_output, enc_output, enc_mask))

    x = self.ln3(x + self.ann(x))
    x = self.dropout(x)
    return x

In [5]:
# class EncoderBlock(nn.Module):
#   def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
#     super().__init__()

#     self.ln1 = nn.LayerNorm(d_model)
#     self.ln2 = nn.LayerNorm(d_model)
#     self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
#     self.ann = nn.Sequential(
#         nn.Linear(d_model, d_model * 4),
#         nn.GELU(),
#         nn.Linear(d_model * 4, d_model),
#         nn.Dropout(dropout_prob)
#     )
#     self.dropout = nn.Dropout(p=dropout_prob)

#   def forward(self, x, mask=None):
#     x = self.ln1(x + self.mha(x, x, x, mask))
#     x = self.ln2(x + self.ann(x))
#     x = self.dropout(x)
#     return x

In [6]:
class MoEEncoderBlock(nn.Module):
    """Encoder block with Mixture of Experts instead of FFN"""
    def __init__(self, d_k, d_model, n_heads, max_len,
                 num_experts=4, top_k=2, dropout_prob=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)

        # Replace standard FFN with MoE
        self.moe = EducationalMoE(d_model, num_experts, top_k)

        self.dropout = nn.Dropout(p=dropout_prob)

    def forward(self, x, mask=None):
        # Self-attention (unchanged)
        x = self.ln1(x + self.mha(x, x, x, mask))

        # MoE feed-forward (instead of regular FFN)
        residual = x
        x = self.ln2(x)
        x = residual + self.dropout(self.moe(x))

        return x

    def get_aux_loss(self):
        return self.moe.get_aux_loss()

In [7]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=2048, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [8]:
# class Encoder(nn.Module):
#   def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers,
#                #n_classes,
#                dropout_prob) -> None:
#     super().__init__()

#     self.embedding = nn.Embedding(vocab_size, d_model)
#     self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
#     transformer_blocks = [
#         EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
#     self.transformer_blocks = nn.Sequential(*transformer_blocks)
#     self.ln = nn.LayerNorm(d_model)
#     #self.fc = nn.Linear(d_model, n_classes)

#   def forward(self, x, mask=None):
#     x = self.embedding(x)
#     x = self.pos_encoding(x)

#     for block in self.transformer_blocks:
#       x = block(x, mask)

#     # many-to-one x has shape (N x T x D) -> (N x D)
#     #x = x[:, 0, :]

#     x = self.ln(x)
#     #x = self.fc(x)
#     return x

In [9]:
# class MoEEncoder(nn.Module):
#     def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers,
#                  num_experts=4, top_k=2, dropout_prob=0.1):
#         super().__init__()

#         self.embedding = nn.Embedding(vocab_size, d_model)
#         self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)

#         # Use MoEEncoderBlock instead of regular EncoderBlock
#         transformer_blocks = [
#             MoEEncoderBlock(d_k, d_model, n_heads, max_len,
#                            num_experts, top_k, dropout_prob)
#             for _ in range(n_layers)
#         ]
#         self.transformer_blocks = nn.Sequential(*transformer_blocks)
#         self.ln = nn.LayerNorm(d_model)

#     def forward(self, x, mask=None):
#         x = self.embedding(x)
#         x = self.pos_encoding(x)

#         for block in self.transformer_blocks:
#           x = block(x, mask)

#         #x = self.transformer_blocks(x, mask)  # MoE blocks!
#         x = self.ln(x)
#         return x

In [10]:
class MoEEncoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers,
                 num_experts=4, top_k=2, dropout_prob=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)

        # Use ModuleList instead of Sequential to access individual blocks
        self.transformer_blocks = nn.ModuleList([
            MoEEncoderBlock(d_k, d_model, n_heads, max_len,
                           num_experts, top_k, dropout_prob)
            for _ in range(n_layers)
        ])
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)

        for block in self.transformer_blocks:
            x = block(x, mask)

        x = self.ln(x)
        return x

    def get_aux_loss(self):
        """Sum auxiliary losses from all MoE blocks"""
        total_aux_loss = torch.tensor(0.0, device=next(self.parameters()).device)
        for block in self.transformer_blocks:
            total_aux_loss = total_aux_loss + block.get_aux_loss()
        return total_aux_loss

In [11]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers, dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    decoder_blocks = [
        DecoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.decoder_blocks = nn.Sequential(*decoder_blocks)
    self.ln = nn.LayerNorm(d_model)
    self.fc = nn.Linear(d_model, vocab_size)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):

    # print("=" * 50)
    # print("DEBUGGING DECODER INPUT:")
    # print(f"dec_input shape: {dec_input.shape}")
    # print(f"dec_input dtype: {dec_input.dtype}")
    # print(f"dec_input min value: {dec_input.min().item()}")
    # print(f"dec_input max value: {dec_input.max().item()}")
    # print(f"Embedding vocab size: {self.embedding.num_embeddings}")
    # print(f"Valid token range: 0 to {self.embedding.num_embeddings - 1}")

    x = self.embedding(dec_input)
    x = self.pos_encoding(x)

    for block in self.decoder_blocks:
      x = block(enc_output, x, enc_mask, dec_mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    # x = x[:, 0, :] - this is for encoder

    x = self.ln(x)
    x = self.fc(x) # many-to-many
    return x

In [12]:
class Transformer(nn.Module):
  def __init__(self, encoder, decoder) -> None:
    super().__init__()

    self.encoder = encoder
    self.decoder = decoder

  def forward(self, enc_input, dec_input, enc_mask, dec_mask):
    enc_output = self.encoder(enc_input, enc_mask)
    dec_output = self.decoder(enc_output, dec_input, enc_mask, dec_mask)

    return dec_output

In [13]:
encoder = MoEEncoder(vocab_size=20_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  num_experts=4,
                  top_k=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=10_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [14]:
# print("Before .to():", encoder.shape, encoder.dtype, encoder.device)
# print("Contains NaN:", torch.isnan(encoder).any())

# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [15]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(10000, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [16]:
xe = np.random.randint(0, 20_000, size=(8, 512))
xe_t = torch.tensor(xe).to(device)

In [17]:
xd = np.random.randint(0, 10_000, size=(8, 256))
xd_t = torch.tensor(xd).to(device)

In [18]:
maske = np.ones((8, 512))
maske[:, 256:] = 0
maske_t = torch.tensor(maske).to(device)

maskd = np.ones((8, 256))
maskd[:, 128:] = 0
maskd_t = torch.tensor(maskd).to(device)

In [19]:
out = transformer(xe_t, xd_t, maske_t, maskd_t)
out.shape

torch.Size([8, 256, 10000])

In [20]:
out

tensor([[[-6.4257e-01, -2.1313e-01, -9.1693e-02,  ..., -1.2741e-03,
           9.3009e-01, -5.4061e-01],
         [-3.3452e-01,  1.0905e+00, -1.3702e-01,  ...,  6.3722e-02,
           6.4387e-01, -4.8773e-01],
         [-4.5714e-02,  2.1257e-02,  4.6108e-01,  ..., -5.0672e-01,
           2.3296e-01, -3.7602e-01],
         ...,
         [-7.7855e-01,  9.7228e-02,  1.0013e+00,  ...,  2.0909e-01,
           2.6029e-01, -4.3840e-01],
         [-3.7762e-01,  2.8666e-01, -5.3592e-01,  ...,  2.6724e-01,
           3.6716e-01,  2.5558e-01],
         [ 2.2218e-01,  3.9916e-02, -8.6191e-01,  ...,  2.8652e-02,
           1.4065e+00, -5.9465e-01]],

        [[ 5.5958e-01, -1.9328e-01,  6.1175e-01,  ..., -1.5961e-01,
          -6.4866e-01, -5.2274e-01],
         [-8.0055e-01,  3.5093e-01, -9.2312e-01,  ..., -5.3488e-01,
          -1.2006e+00, -5.0670e-02],
         [ 5.0655e-01,  1.1009e+00, -1.4166e-01,  ...,  1.2126e-01,
           1.0531e+00, -4.0616e-01],
         ...,
         [-2.0401e-01,  2

In [21]:
!head rus.txt

Go.	Марш!	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1159202 (shanghainese)
Go.	Иди.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898247 (marafon)
Go.	Идите.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898250 (marafon)
Hi.	Здравствуйте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #402127 (odexed)
Hi.	Привет!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #466968 (katjka)
Hi.	Хай.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #467233 (timsa)
Hi.	Здрасте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3803577 (marafon)
Hi.	Здоро́во!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3854188 (marafon)
Hi.	Приветик!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #7234283 (marafon)
Run!	Беги!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1569978 (Biga)


In [22]:
import pandas as pd

df = pd.read_csv('rus.txt', sep="\t", header=None)
df.head()

,0,1,2
0,Go.,Марш!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Иди.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Идите.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Hi.,Здравствуйте.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Привет!,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [23]:
df.shape

(34723, 3)

In [24]:
df = df.iloc[:30_000]

In [25]:
df.columns = ['en', 'ru', 'd']

In [26]:
df

,en,ru,d
0,Go.,Марш!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Иди.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Идите.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Hi.,Здравствуйте.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Привет!,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
...,...,...,...
29995,Tom is European.,Том - европеец.,CC-BY 2.0 (France) Attribution: tatoeba.org #1...
29996,Tom is a Cancer.,Том - Рак.,CC-BY 2.0 (France) Attribution: tatoeba.org #7...
29997,Tom is a Gemini.,Том – Близнецы.,CC-BY 2.0 (France) Attribution: tatoeba.org #7...
29998,Tom is a Muslim.,Том - мусульманин.,CC-BY 2.0 (France) Attribution: tatoeba.org #1...


In [27]:
df = df.drop('d', axis=1)

In [28]:
df

,en,ru
0,Go.,Марш!
1,Go.,Иди.
2,Go.,Идите.
3,Hi.,Здравствуйте.
4,Hi.,Привет!
...,...,...
29995,Tom is European.,Том - европеец.
29996,Tom is a Cancer.,Том - Рак.
29997,Tom is a Gemini.,Том – Близнецы.
29998,Tom is a Muslim.,Том - мусульманин.


In [29]:
df.to_csv('rus.csv', index=None)

In [30]:
!head rus.csv

en,ru
Go.,Марш!
Go.,Иди.
Go.,Идите.
Hi.,Здравствуйте.
Hi.,Привет!
Hi.,Хай.
Hi.,Здрасте.
Hi.,Здоро́во!
Hi.,Приветик!


In [31]:
!pip install transformers datasets sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 23.1 MB/s eta 0:00:00


In [32]:
from datasets import load_dataset
raw_dataset = load_dataset('csv', data_files='rus.csv')

Generating train split: 0 examples [00:00, ? examples/s]

In [33]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 30000
    })
})

In [34]:
split = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 9000
    })
})

In [35]:
from transformers import AutoTokenizer

model_checkpoint = 'Helsinki-NLP/opus-mt-en-ru'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

In [36]:
en_sentence = split['train'][0]['en']
ru_sentence = split['train'][0]['ru']

inputs = tokenizer(en_sentence)
targets = tokenizer(text_target=ru_sentence)

tokenizer.convert_ids_to_tokens(targets['input_ids'])

['▁Не', '▁', 'двига', 'й', 'ся', '!', '</s>']

In [37]:
en_sentence, ru_sentence

("Don't move!", 'Не двигайся!')

In [38]:
max_input_length = 128
max_target_length = 128

In [39]:
def preprocess_function(batch):
  model_inputs = tokenizer(batch['en'], max_length=max_input_length, truncation=True)
  labels = tokenizer(text_target=batch['ru'], max_length=max_target_length, truncation=True)

  model_inputs['labels'] = labels['input_ids']
  return model_inputs

In [40]:
tokenized_datasets = split.map(preprocess_function, batched=True, remove_columns=split['train'].column_names)

Map:   0%|          | 0/21000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

In [41]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9000
    })
})

In [42]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer)

In [43]:
batch = data_collator([tokenized_datasets['train'][i] for i in range(0, 5)])
batch.keys()

KeysView({'input_ids': tensor([[ 1519,    18,    45,  2728,    56,     0, 62517],
        [11875,    55, 50869,     3,     0, 62517, 62517],
        [  961,    33,   182,   108,    19,     0, 62517],
        [  103,    18,   153,   195,  2007,     3,     0],
        [  160,  4894, 13617,     3,     0, 62517, 62517]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]]), 'labels': tensor([[  327,    21,  9015,    48,    93,    56,     0],
        [31836,  1723,  4029,   387,     3,     0,  -100],
        [  784,   247,  2945,   224,    19,     0,  -100],
        [  206,  3211,  2920,   106,     3,     0,  -100],
        [  261,   907, 13239,   357,  4047,     3,     0]])})

In [44]:
batch['input_ids']

tensor([[ 1519,    18,    45,  2728,    56,     0, 62517],
        [11875,    55, 50869,     3,     0, 62517, 62517],
        [  961,    33,   182,   108,    19,     0, 62517],
        [  103,    18,   153,   195,  2007,     3,     0],
        [  160,  4894, 13617,     3,     0, 62517, 62517]])

In [45]:
batch['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])

In [46]:
batch['labels']

tensor([[  327,    21,  9015,    48,    93,    56,     0],
        [31836,  1723,  4029,   387,     3,     0,  -100],
        [  784,   247,  2945,   224,    19,     0,  -100],
        [  206,  3211,  2920,   106,     3,     0,  -100],
        [  261,   907, 13239,   357,  4047,     3,     0]])

In [47]:
tokenizer.all_special_ids

[0, 1, 62517]

In [48]:
tokenizer.all_special_tokens

['</s>', '<unk>', '<pad>']

In [49]:
tokenizer('<pad>')

{'input_ids': [62517, 0], 'attention_mask': [1, 1]}

In [50]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

valid_loader = DataLoader(
    tokenized_datasets['test'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

In [51]:
for batch in train_loader:
  for k, v in batch.items():
    print("k: ", k, "v.shape: ", v.shape)
  break

k:  input_ids v.shape:  torch.Size([32, 8])
k:  attention_mask v.shape:  torch.Size([32, 8])
k:  labels v.shape:  torch.Size([32, 11])


In [52]:
tokenizer.vocab_size

62518

In [53]:
tokenizer.decode(62517)

'<pad>'

In [54]:
tokenizer.add_special_tokens({"cls_token": "<s>"})

1

In [55]:
tokenizer('<s>')

{'input_ids': [62518, 0], 'attention_mask': [1, 1]}

In [56]:
tokenizer.decode(tokenizer('<s>')['input_ids'])

'<s> </s>'

In [57]:
tokenizer.vocab_size

62518

In [58]:
encoder = MoEEncoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  num_experts=4,
                  top_k=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [59]:
# torch.cuda.empty_cache()

In [60]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(62519, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [61]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(transformer.parameters())

In [62]:
# from datetime import datetime

# def train(model, criterion, optimizer, train_loader, valid_loader, epochs):
#   train_losses = np.zeros(epochs)
#   test_losses = np.zeros(epochs)

#   for it in range(epochs):
#     model.train()
#     t0 = datetime.now()
#     train_loss = []
#     for batch in train_loader:
#       # move data to GPU
#       batch = {k: v.to(device) for k, v in batch.items()}

#       # zero the parameter gradients
#       optimizer.zero_grad()

#       enc_input = batch['input_ids']
#       enc_mask = batch['attention_mask']
#       targets = batch['labels']

#       # shift targets forwards to get decoder_input
#       dec_input = targets.clone().detach()
#       dec_input = torch.roll(dec_input, shifts=1, dims=1)
#       dec_input[:, 0] = 62_518 #65_001

#       # also convert all -100 to pad token id
#       dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

#       # make decoder input mask
#       dec_mask = torch.ones_like(dec_input)
#       dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


#       # forward pass
#       outputs = model(enc_input, dec_input, enc_mask, dec_mask)
#       loss = criterion(outputs.transpose(2, 1), targets)

#       # backward and optimizer
#       loss.backward()
#       optimizer.step()
#       train_loss.append(loss.item())

#     # get average train loss
#     train_loss = np.mean(train_loss)

#     model.eval()
#     test_loss = []


#     for batch in valid_loader:
#       # move data to GPU
#       batch = {k: v.to(device) for k, v in batch.items()}

#       # zero the parameter gradients
#       # optimizer.zero_grad()

#       enc_input = batch['input_ids']
#       enc_mask = batch['attention_mask']
#       targets = batch['labels']

#       # shift targets forwards to get decoder_input
#       dec_input = targets.clone().detach()
#       dec_input = torch.roll(dec_input, shifts=1, dims=1)
#       dec_input[:, 0] = 62_518 #65_001

#       # also convert all -100 to pad token id
#       dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

#       # make decoder input mask
#       dec_mask = torch.ones_like(dec_input)
#       dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


#       # forward pass
#       outputs = model(enc_input, dec_input, enc_mask, dec_mask)
#       loss = criterion(outputs.transpose(2, 1), targets)

#       # backward and optimizer
#       # loss.backward()
#       # optimizer.step()
#       test_loss.append(loss.item())

#     # get average train loss
#     test_loss = np.mean(test_loss)



#     train_losses[it] = train_loss
#     test_losses[it] = test_loss

#     dt = datetime.now() - t0
#     print(f'Epoch {it+1}/{epochs}, Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, Duration: {dt}')

#   return train_losses, test_losses

In [63]:
from datetime import datetime

def train(model, criterion, optimizer, train_loader, valid_loader, epochs, aux_loss_weight=0.01):
    train_losses = np.zeros(epochs)
    test_losses = np.zeros(epochs)

    for it in range(epochs):
        model.train()
        t0 = datetime.now()
        train_loss = []

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()

            enc_input = batch['input_ids']
            enc_mask = batch['attention_mask']
            targets = batch['labels']

            # Shift targets for decoder
            dec_input = targets.clone().detach()
            dec_input = torch.roll(dec_input, shifts=1, dims=1)
            dec_input[:, 0] = 62_518
            dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)
            dec_mask = torch.ones_like(dec_input)
            dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)

            # Forward pass
            outputs = model(enc_input, dec_input, enc_mask, dec_mask)

            # Main cross-entropy loss
            main_loss = criterion(outputs.transpose(2, 1), targets)

            # ADD THIS: Get auxiliary loss from encoder's MoE layers
            aux_loss = model.encoder.get_aux_loss()

            # Combined loss
            total_loss = main_loss + aux_loss_weight * aux_loss

            # Backward pass
            total_loss.backward()
            optimizer.step()

            train_loss.append(total_loss.item())

        # get average train loss
        train_loss = np.mean(train_loss)

        # Validation (no aux loss needed)
        model.eval()
        test_loss = []
        for batch in valid_loader:
          # move data to GPU
          batch = {k: v.to(device) for k, v in batch.items()}

          # zero the parameter gradients
          # optimizer.zero_grad()

          enc_input = batch['input_ids']
          enc_mask = batch['attention_mask']
          targets = batch['labels']

          # shift targets forwards to get decoder_input
          dec_input = targets.clone().detach()
          dec_input = torch.roll(dec_input, shifts=1, dims=1)
          dec_input[:, 0] = 62_518 #65_001

          # also convert all -100 to pad token id
          dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

          # make decoder input mask
          dec_mask = torch.ones_like(dec_input)
          dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


          # forward pass
          outputs = model(enc_input, dec_input, enc_mask, dec_mask)
          loss = criterion(outputs.transpose(2, 1), targets)

          # backward and optimizer
          # loss.backward()
          # optimizer.step()
          test_loss.append(loss.item())

        # get average train loss
        test_loss = np.mean(test_loss)

        train_losses[it] = train_loss
        test_losses[it] = test_loss

        dt = datetime.now() - t0

        # Print losses
        print(f'Epoch {it+1}/{epochs}, Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}')

    return train_losses, test_losses

In [64]:
train_losses, test_losses = train(transformer, criterion, optimizer, train_loader, valid_loader, epochs=15)

Epoch 1/15, Train loss: 4.6321, Test loss: 3.5877
Epoch 2/15, Train loss: 3.3127, Test loss: 2.9648
Epoch 3/15, Train loss: 2.7910, Test loss: 2.5891
Epoch 4/15, Train loss: 2.4166, Test loss: 2.3391
Epoch 5/15, Train loss: 2.1369, Test loss: 2.1708
Epoch 6/15, Train loss: 1.9291, Test loss: 2.0574
Epoch 7/15, Train loss: 1.7634, Test loss: 1.9805
Epoch 8/15, Train loss: 1.6338, Test loss: 1.9317
Epoch 9/15, Train loss: 1.5265, Test loss: 1.8837
Epoch 10/15, Train loss: 1.4449, Test loss: 1.8530
Epoch 11/15, Train loss: 1.3746, Test loss: 1.8282
Epoch 12/15, Train loss: 1.3043, Test loss: 1.8127
Epoch 13/15, Train loss: 1.2587, Test loss: 1.7996
Epoch 14/15, Train loss: 1.2078, Test loss: 1.7900
Epoch 15/15, Train loss: 1.1690, Test loss: 1.7873


In [65]:
def check_expert_balance(model, dataloader, num_batches=5):
    model.eval()

    # Get first MoE block from encoder
    first_moe_block = model.encoder.transformer_blocks[0]
    num_experts = first_moe_block.moe.num_experts

    usage_counts = {i: 0 for i in range(num_experts)}
    total_tokens = 0

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_batches:
                break

            batch = {k: v.to(device) for k, v in batch.items()}
            enc_input = batch['input_ids']
            enc_mask = batch['attention_mask']

            # Get router probabilities from first MoE block
            x = model.encoder.embedding(enc_input)
            x = model.encoder.pos_encoding(x)

            # Pass through first MoE block only to get router decisions
            block = model.encoder.transformer_blocks[0]

            # Get router logits
            router_logits = block.moe.router(x)
            router_probs = F.softmax(router_logits, dim=-1)
            top_k_probs, top_k_indices = torch.topk(router_probs, block.moe.top_k, dim=-1)

            # Count expert usage
            for expert_idx in range(num_experts):
                usage_counts[expert_idx] += (top_k_indices == expert_idx).sum().item()
            total_tokens += enc_input.numel() * block.moe.top_k  # Multiply by top_k because each token uses top_k experts

    # Print distribution
    print(f"\nExpert utilization (based on {num_batches} batches):")
    print(f"Total expert assignments: {total_tokens}")
    print("-" * 40)

    for expert_idx in range(num_experts):
        percentage = 100 * usage_counts[expert_idx] / total_tokens
        bars = "█" * int(percentage / 2)
        print(f"  Expert {expert_idx}: {percentage:5.1f}% {bars} ({usage_counts[expert_idx]} assignments)")

    return usage_counts

# Call it after your training
expert_usage = check_expert_balance(transformer, train_loader, num_batches=10)


Expert utilization (based on 10 batches):
Total expert assignments: 5120
----------------------------------------
  Expert 0:  29.4% ██████████████ (1503 assignments)
  Expert 1:  26.2% █████████████ (1341 assignments)
  Expert 2:  17.2% ████████ (880 assignments)
  Expert 3:  27.3% █████████████ (1396 assignments)


In [66]:
input_sentence = split['test'][10]['en']
input_sentence

'Tom stood, too.'

In [67]:
enc_input = tokenizer(input_sentence, return_tensors='pt')
enc_input

{'input_ids': tensor([[ 2226, 13269,     2,   846,     3,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [68]:
dec_input_str = '<s>'

dec_input = tokenizer(text_target=dec_input_str, return_tensors='pt')
dec_input

{'input_ids': tensor([[62518,     0]]), 'attention_mask': tensor([[1, 1]])}

In [69]:
enc_input.to(device)
dec_input.to(device)
output = transformer(enc_input['input_ids'],
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

output

tensor([[[ -1.0827, -10.4279,   5.8632,  ...,  -9.4827, -10.7507,  -8.7433]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [70]:
output.shape # N x T x V

torch.Size([1, 1, 62519])

In [71]:
enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])
enc_output.shape

torch.Size([1, 6, 64])

In [72]:
dec_output = decoder(enc_output,
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

dec_output.shape

torch.Size([1, 1, 62519])

In [73]:
torch.allclose(output, dec_output)

True

In [74]:
dec_input_ids = dec_input['input_ids'][:, :-1]
dec_attn_mask = dec_input['attention_mask'][:, :-1]

for _ in range(32):
  dec_output = decoder(enc_output,
                      dec_input_ids,
                      enc_input['attention_mask'],
                      dec_attn_mask)

  # choose the best value (or sample)
  prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

  # append to decoder input
  dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

  # recreate mask
  dec_attn_mask = torch.ones_like(dec_input_ids)

  # exit when reach </s>
  if prediction_id == 0:
    break


In [75]:
tokenizer.decode(dec_input_ids[0])

'<s>  Том тоже пошёл.</s>'

In [76]:
split['test'][10]['ru']

'Том тоже стоял.'

In [77]:
def greedy_translate(input_sentence):
  # get encoder output first
  enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
  enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

  # setup initial decoder input
  #dec_input_ids = torch.tensor([[65_001]], device=device)
  dec_input_ids = torch.tensor([[62_518]], device=device)
  dec_attn_mask = torch.ones_like(dec_input_ids, device=device)

  # now do the decoder loop
  for _ in range(32):
    dec_output = decoder(enc_output,
                        dec_input_ids,
                        enc_input['attention_mask'],
                        dec_attn_mask)

    # choose the best value (or sample)
    prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

    # append to decoder input
    dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

    # recreate mask
    dec_attn_mask = torch.ones_like(dec_input_ids)

    # exit when reach </s>
    if prediction_id == 0:
      break

  translation = tokenizer.decode(dec_input_ids[0, 1:])
  return translation

In [78]:
def beam_search_translate(input_sentence, beam_width=5, max_len=32, length_penalty=0.6):
    """
    Translate using beam search decoding.

    Args:
        input_sentence: English sentence to translate
        beam_width: Number of hypotheses to keep (typically 3-10)
        max_len: Maximum translation length
        length_penalty: Alpha parameter (0 = no penalty, 0.6-0.7 is common)

    Returns:
        Best translation string and its score
    """
    transformer.eval()

    # Encode the input sentence once (shared for all beams)
    enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
    enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

    # Initialize beams: (sequence, log_probability)
    start_token = torch.tensor([[tokenizer.cls_token_id]], device=device)
    beams = [(start_token, 0.0)]  # (decoder_input_ids, cumulative_log_prob)
    completed_beams = []  # Store finished translations

    for step in range(max_len):
        all_candidates = []

        # Expand each beam
        for seq, score in beams:
            # Check if this beam is already complete
            if seq[0, -1].item() == tokenizer.eos_token_id:  # 0 is </s>
                completed_beams.append((seq, score))
                continue

            # Create attention mask for this beam
            dec_attn_mask = torch.ones_like(seq)

            # Get model predictions for the next token
            with torch.no_grad():
                dec_output = decoder(
                    enc_output,
                    seq,
                    enc_input['attention_mask'],
                    dec_attn_mask
                )

            # Get log probabilities for the last token
            last_token_logits = dec_output[:, -1, :]  # (1, vocab_size)
            log_probs = F.log_softmax(last_token_logits, dim=-1)  # Convert to log space

            # Get top k candidates (beam_width * 2 to maintain diversity)
            top_k_log_probs, top_k_indices = torch.topk(log_probs, beam_width * 2, dim=-1)

            # Create new candidates
            for i in range(beam_width * 2):
                new_token = top_k_indices[0, i].unsqueeze(0).unsqueeze(0)  # (1, 1)
                new_seq = torch.cat([seq, new_token], dim=-1)
                new_score = score + top_k_log_probs[0, i].item()  # Add log prob
                all_candidates.append((new_seq, new_score))

        # If all beams are complete, break early
        if not all_candidates and completed_beams:
            break

        # Select top k beams based on score
        all_candidates.sort(key=lambda x: x[1], reverse=True)  # Higher log prob is better
        beams = all_candidates[:beam_width]

    # Add any completed beams from the loop
    beams.extend(completed_beams)

    # Apply length penalty to avoid favoring shorter translations
    def apply_length_penalty(seq, score):
        """Normalize score by sequence length (excluding start token)"""
        length = seq.shape[1] - 1  # Exclude start token
        if length == 0:
            return score
        # Standard length penalty from Transformer paper
        return score / (length ** length_penalty)

    # Choose the best beam using length penalty
    best_seq, best_score = max(beams, key=lambda x: apply_length_penalty(x[0], x[1]))

    # Decode and return (skip the start token)
    translation = tokenizer.decode(best_seq[0, 1:])
    return translation, best_score

In [79]:
# Test sentences
test_sentences = [
    "How are you?",
    "I love you",
    "Good morning",
    "Thank you",
    "Where is the hotel?",
    "I would like to order coffee"
]

# Compare greedy vs beam search
print("COMPARING GREEDY VS BEAM SEARCH")
print("=" * 60)

for sentence in test_sentences:
    print(f"\nEnglish: {sentence}")
    print("-" * 40)

    # Greedy
    greedy_result = greedy_translate(sentence)
    print(f"Greedy:     {greedy_result}")

    # Beam search with width 3
    beam_result, score = beam_search_translate(sentence, beam_width=3)
    print(f"Beam (b=3): {beam_result} (score: {score:.2f})")

    # Beam search with width 5 (slower but potentially better)
    beam_result_5, score_5 = beam_search_translate(sentence, beam_width=5)
    print(f"Beam (b=5): {beam_result_5} (score: {score_5:.2f})")

# See the search process for one sentence
# beam_search_verbose("I love you", beam_width=3)

COMPARING GREEDY VS BEAM SEARCH

English: How are you?
----------------------------------------
Greedy:     Как ты?</s>
Beam (b=3): Как ты?</s> (score: -1.72)
Beam (b=5): Как ты?</s> (score: -1.72)

English: I love you
----------------------------------------
Greedy:     Я вас люблю.</s>
Beam (b=3): Я вас люблю.</s> (score: -2.09)
Beam (b=5): Я вас люблю.</s> (score: -2.09)

English: Good morning
----------------------------------------
Greedy:     Стыдно.</s>
Beam (b=3): Стыдно.</s> (score: -3.10)
Beam (b=5): Стыдно.</s> (score: -3.10)

English: Thank you
----------------------------------------
Greedy:     Спасибо тебе.</s>
Beam (b=3): Спасибо тебе.</s> (score: -1.25)
Beam (b=5): Спасибо тебе.</s> (score: -1.25)

English: Where is the hotel?
----------------------------------------
Greedy:     Где смотрит?</s>
Beam (b=3): Где платит?</s> (score: -2.59)
Beam (b=5): Где платит?</s> (score: -2.59)

English: I would like to order coffee
----------------------------------------
Greedy:   

In [80]:
# res = greedy_translate("How are you?")
# res
sentence = "How are you?"
greedy_result = greedy_translate(sentence)
greedy_result
print(f"Greedy:     {greedy_result}")

Greedy:     Как ты?</s>
